In [230]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression 
# *** ALTERAÇÃO ***: Importar GridSearchCV E TimeSeriesSplit
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit 
import warnings
import time 

warnings.filterwarnings('ignore', category=UserWarning, module='xgboost')
warnings.filterwarnings('ignore', category=Warning) 

print("--- [Início do Pipeline (v33 - Otimização Temporal Robusta)] ---")
print("A otimizar (GridSearch) o Stacking (v16) usando TimeSeriesSplit...")

# --- 1. FUNÇÃO DE PRÉ-PROCESSAMENTO (IDÊNTICA AO v16/v32) ---

def formatar_celula(series_coluna):
    s = series_coluna.astype(str).replace('NULL', pd.NA)
    s = s.str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8')
    s = s.str.upper()
    s = s.str.replace(r'[^A-Z0-9]+', '_', regex=True)
    s = s.str.strip('_')
    s = s.replace('', pd.NA)
    return s

def preprocessar_dados(df, colunas_scaler_treinadas=None, scaler=None):
    # Retorna df_final_row_ids para o teste
    if 'RowId' not in df.columns and 'AVERAGE_SPEED_DIFF' not in df.columns:
        df_final_row_ids = np.arange(1, len(df) + 1)
    else:
        df_final_row_ids = None # Não precisamos no treino

    cols_to_transform = ['AVERAGE_CLOUDINESS', 'AVERAGE_RAIN']
    for col in cols_to_transform:
        if col in df.columns:
            df[col] = formatar_celula(df[col])

    # Guardar record_date antes de a remover (PRECISAMOS DELA PARA ORDENAR!)
    record_date_col = None
    if 'record_date' in df.columns:
        try:
            record_date_col = pd.to_datetime(df['record_date'], format='mixed', dayfirst=True)
        except Exception:
             record_date_col = None 

    cols_to_drop_base = ['city_name', 'AVERAGE_RAIN', 'AVERAGE_PRECIPITATION', 'record_date']
    
    if record_date_col is not None:
        df['Hora_sin'] = np.sin(2 * np.pi * record_date_col.dt.hour/24)
        df['Hora_cos'] = np.cos(2 * np.pi * record_date_col.dt.hour/24)
        df['Mes_sin'] = np.sin(2 * np.pi * record_date_col.dt.month/12)
        df['Mes_cos'] = np.cos(2 * np.pi * record_date_col.dt.month/12)
        df['DIA_SEMANA'] = record_date_col.dt.dayofweek
        df['IS_WEEKEND'] = df['DIA_SEMANA'].isin([5, 6]).astype(int)
        rush_hours = [7, 8, 9, 17, 18, 19]
        df['IS_RUSH_HOUR'] = record_date_col.dt.hour.isin(rush_hours).astype(int)
        
    cols_existentes_drop = [col for col in cols_to_drop_base if col in df.columns and col != 'record_date'] 
    df = df.drop(columns=cols_existentes_drop)
    # Remover record_date só depois de ordenar e usar fora da função

    if 'AVERAGE_CLOUDINESS' in df.columns:
        df['AVERAGE_CLOUDINESS'] = df['AVERAGE_CLOUDINESS'].replace('NAN', np.nan)
        df['AVERAGE_CLOUDINESS'] = df['AVERAGE_CLOUDINESS'].fillna('NONE')

    cols_to_onehot = ['LUMINOSITY', 'AVERAGE_CLOUDINESS', 'DIA_SEMANA']
    for col in cols_to_onehot:
        if col in df.columns:
            prefix = col[:3].upper()
            if col == 'DIA_SEMANA': prefix = 'DAY'
            dummies = pd.get_dummies(df[col], prefix=prefix, dtype=int, sparse=False)
            df = pd.concat([df, dummies], axis=1)
            df = df.drop(col, axis=1)

    cols_to_normalize = [
        'AVERAGE_FREE_FLOW_SPEED', 'AVERAGE_TIME_DIFF', 'AVERAGE_FREE_FLOW_TIME',
        'AVERAGE_TEMPERATURE', 'AVERAGE_ATMOSP_PRESSURE', 'AVERAGE_HUMIDITY',
        'AVERAGE_WIND_SPEED', 'IS_WEEKEND', 'IS_RUSH_HOUR',
        'Hora_sin', 'Hora_cos', 'Mes_sin', 'Mes_cos'
    ]
    cols_existentes_normalize = [col for col in cols_to_normalize if col in df.columns]
    
    if scaler is None:
        #print("... (TREINO GRID TEMPORAL) A fitar o MinMaxScaler.")
        scaler = MinMaxScaler()
        if cols_existentes_normalize:
            df[cols_existentes_normalize] = scaler.fit_transform(df[cols_existentes_normalize])
        return df, scaler, cols_existentes_normalize, df_final_row_ids # Retornar ids no treino final
    else:
        #print("... (VALIDAÇÃO GRID TEMPORAL) A aplicar o MinMaxScaler já treinado.")
        cols_para_scaler_teste = [col for col in colunas_scaler_treinadas if col in df.columns]
        if cols_para_scaler_teste:
            df[cols_para_scaler_teste] = scaler.transform(df[cols_para_scaler_teste])
        # Retornar ids também no teste (embora não usados aqui, boa prática)
        return df, scaler, colunas_scaler_treinadas, df_final_row_ids 


# --- 2. CARREGAR E ORDENAR DADOS ---
print("\n--- [Passo 1/6] A carregar e ORDENAR dados de Treino... ---")
# Ler sem forçar parse_dates (nem todos os ficheiros têm a coluna 'record_date')
# Tentativa robusta: tentar vários separadores. Muitos erros KeyError('AVERAGE_SPEED_DIFF')
# vêm de leitura com separador errado (ficheiro lido como 1 coluna).
try:
    df_train = pd.read_csv("datasets/training_data.csv", delimiter=";", encoding="latin-1")
    # Se acabou por ler apenas 1 coluna, tentar autodetecção / separador vírgula
    if df_train.shape[1] == 1:
        try:
            df_train = pd.read_csv("datasets/training_data.csv", sep=None, engine='python', encoding="latin-1")
        except Exception:
            df_train = pd.read_csv("datasets/training_data.csv", sep=',', engine='python', encoding="latin-1")
except Exception:
    # Último recurso: forçar separador vírgula
    df_train = pd.read_csv("datasets/training_data.csv", sep=',', engine='python', encoding="latin-1")

# Tentar converter a coluna 'record_date' se existir (não falhar se a coluna estiver ausente)
if 'record_date' in df_train.columns:
    df_train['record_date'] = pd.to_datetime(df_train['record_date'], dayfirst=True, errors='coerce')
    # Se a conversão teve sucesso em pelo menos uma linha, ordenar por record_date
    if df_train['record_date'].notna().any():
        df_train = df_train.sort_values(by='record_date').reset_index(drop=True)
    else:
        df_train = df_train.reset_index(drop=True)
else:
    df_train = df_train.reset_index(drop=True)

# Verificação extra: garantir que a coluna alvo exista — ajuda a diagnosticar problemas cedo
if 'AVERAGE_SPEED_DIFF' not in df_train.columns:
    raise KeyError(
        "Coluna 'AVERAGE_SPEED_DIFF' não encontrada em df_train. "
        "Verifique o separador e o conteúdo de 'datasets/training_data.csv'. "
        f"Colunas lidas: {list(df_train.columns)[:20]}"
    )

print("... Dados de treino ordenados.")

y_train_raw = df_train['AVERAGE_SPEED_DIFF']
le = LabelEncoder()
y_train_formatado = formatar_celula(y_train_raw).replace('NAN', 'NONE').fillna('NONE')
y_train_encoded = le.fit_transform(y_train_formatado)
print(f"Classes do Alvo encontradas (em MAIÚSCULAS): {list(le.classes_)}")

# --- 3. PRÉ-PROCESSAMENTO (NOS DADOS ORDENADOS) ---
print("--- [Passo 2/6] A pré-processar os dados de TREINO ORDENADOS... ---")
# Processar os dados e guardar scaler/colunas para o teste final
X_train_processed, scaler_final, colunas_scaler_final, _ = preprocessar_dados(df_train.drop(columns=['AVERAGE_SPEED_DIFF']))

# Remover record_date APÓS processamento
if 'record_date' in X_train_processed.columns:
    X_train_processed = X_train_processed.drop(columns=['record_date'])
X_train_processed = X_train_processed.fillna(0) 
print(f"Shape final do Treino (X_train): {X_train_processed.shape}") # Deve ser 33

# --- 4. CONFIGURAR MODELOS E GRIDSEARCH ---
print("--- [Passo 3/6] A configurar Stacking e GridSearchCV Temporal... ---")

# Modelos Base (com nomes únicos para o GridSearch)
base_xgb = XGBClassifier(
    subsample=0.7, colsample_bytree=0.9, # Fixos do v16
    use_label_encoder=False, eval_metric='mlogloss',
    random_state=42, n_jobs=1 # n_jobs=1 aqui para não conflitar com GridSearch
)
base_lgbm = LGBMClassifier(
    subsample=0.9, colsample_bytree=0.8, # Fixos do v16
    random_state=42, n_jobs=1, verbose=-1 # n_jobs=1 aqui
)
# Meta Modelo
meta_lr = LogisticRegression(random_state=42, n_jobs=1) # n_jobs=1 aqui

# O Stacking Classifier
# IMPORTANTE: Definir os estimadores aqui, mas os parâmetros serão ajustados pelo GridSearchCV
stack_clf_base = StackingClassifier(
    estimators=[('xgb', base_xgb), ('lgbm', base_lgbm)], 
    final_estimator=meta_lr, 
    cv=5, # CV interno do Stacking
    n_jobs=1 # n_jobs=1 aqui
)

# Definir a Grelha de Parâmetros
# Usar __ para aceder aos parâmetros dos modelos base
param_grid = {
    'xgb__learning_rate': [0.08, 0.1], # Testar à volta do 0.08 do v11
    'xgb__max_depth': [5, 6],         # Testar à volta do 5 do v11
    'xgb__n_estimators': [100, 120],  # Testar à volta do 100 do v11
    
    'lgbm__learning_rate': [0.01, 0.05],# Testar à volta do 0.01 do v16
    'lgbm__num_leaves': [40, 50],     # Testar à volta do 50 do v16
    'lgbm__n_estimators': [200, 250], # Testar à volta do 200 do v16
    
    # Podemos até otimizar o meta-modelo (ex: regularização C)
    'final_estimator__C': [0.1, 1.0, 10.0] 
}
# Total de combinações: 2*2*2 * 2*2*2 * 3 = 8 * 8 * 3 = 192 combinações

# Definir a Estratégia de Validação Temporal
n_splits = 5
tscv = TimeSeriesSplit(n_splits=n_splits)

# Configurar o GridSearchCV
grid_search_ts = GridSearchCV(
    estimator=stack_clf_base,
    param_grid=param_grid,
    cv=tscv, # <<<=== AQUI ESTÁ A MAGIA
    scoring='accuracy',
    n_jobs=-1, # Usar todos os cores para o GridSearch
    verbose=2 # Mostrar mais detalhes do progresso
)

# --- 5. EXECUTAR A OTIMIZAÇÃO TEMPORAL ---
print("\n--- [Passo 4/6] A EXECUTAR o GridSearchCV com TimeSeriesSplit... ---")
print(f"A testar {len(param_grid['xgb__learning_rate']) * len(param_grid['xgb__max_depth']) * len(param_grid['xgb__n_estimators']) * len(param_grid['lgbm__learning_rate']) * len(param_grid['lgbm__num_leaves']) * len(param_grid['lgbm__n_estimators']) * len(param_grid['final_estimator__C'])} combinações...")
print("ISTO VAI DEMORAR MUITO TEMPO!")

start_time = time.time()
grid_search_ts.fit(X_train_processed, y_train_encoded)
end_time = time.time()

print(f"\nOtimização Temporal concluída em {(end_time - start_time) / 60:.2f} minutos.")
print(f"MELHOR SCORE (CV Temporal Robusto) encontrado: {grid_search_ts.best_score_:.5f}")
print("Melhores Parâmetros encontrados (otimizados para o futuro):")
print(grid_search_ts.best_params_)

# Guardar o melhor estimador encontrado
best_stack_ts = grid_search_ts.best_estimator_

print("... A processar dados de Teste.")
# Ler sem forçar parse_dates (evita erro se record_date não existir)
df_test = pd.read_csv("datasets/test_data.csv", delimiter=",", encoding="latin-1")
# Tentar converter a coluna 'record_date' se existir
if 'record_date' in df_test.columns:
    df_test['record_date'] = pd.to_datetime(df_test['record_date'], dayfirst=True, errors='coerce')
test_row_ids = np.arange(1, len(df_test) + 1)
# Usar o scaler_final e colunas_scaler_final do treino completo
X_test, _, _, _= preprocessar_dados(df_test, colunas_scaler_treinadas=colunas_scaler_final, scaler=scaler_final) 
best_stack_ts.fit(X_train_processed, y_train_encoded) 
print("... Modelo final treinado.")

print("--- [Passo 6/6] A gerar previsões e a criar submission.csv... ---")
# Carregar e processar Teste
print("... A processar dados de Teste.")
df_test = pd.read_csv("datasets/test_data.csv", delimiter=",", encoding="latin-1", parse_dates=['record_date'], dayfirst=True)
test_row_ids = np.arange(1, len(df_test) + 1)
# Usar o scaler_final e colunas_scaler_final do treino completo
X_test, _, _, _= preprocessar_dados(df_test, colunas_scaler_treinadas=colunas_scaler_final, scaler=scaler_final) 
if 'record_date' in X_test.columns:
    X_test = X_test.drop(columns=['record_date'])
X_test = X_test.reindex(columns=X_train_processed.columns, fill_value=0)
X_test = X_test.fillna(0)

# Prever com o melhor modelo temporal
y_pred_encoded = best_stack_ts.predict(X_test)
y_pred_labels_upper = le.inverse_transform(y_pred_encoded)
y_pred_labels_final = pd.Series(y_pred_labels_upper).str.title()
submission_df = pd.DataFrame({'RowId': test_row_ids, 'Speed_Diff': y_pred_labels_final})
submission_df.to_csv('submission_v33_TimeSeriesOptimized.csv', index=False)
print("Ficheiro 'submission_v33_TimeSeriesOptimized.csv' criado com sucesso!")

print("\n--- [FIM DO PIPELINE DE OTIMIZAÇÃO TEMPORAL] ---")
print(f"Score CV Aleatório (v16): 0.81489 (Kaggle: 0.83555)")
print(f"Score CV TEMPORAL OTIMIZADO (v33): {grid_search_ts.best_score_:.5f}")

--- [Início do Pipeline (v33 - Otimização Temporal Robusta)] ---
A otimizar (GridSearch) o Stacking (v16) usando TimeSeriesSplit...

--- [Passo 1/6] A carregar e ORDENAR dados de Treino... ---
... Dados de treino ordenados.
Classes do Alvo encontradas (em MAIÚSCULAS): ['HIGH', 'LOW', 'MEDIUM', 'NONE', 'VERY_HIGH']
--- [Passo 2/6] A pré-processar os dados de TREINO ORDENADOS... ---
Shape final do Treino (X_train): (6812, 33)
--- [Passo 3/6] A configurar Stacking e GridSearchCV Temporal... ---

--- [Passo 4/6] A EXECUTAR o GridSearchCV com TimeSeriesSplit... ---
A testar 192 combinações...
ISTO VAI DEMORAR MUITO TEMPO!
Fitting 5 folds for each of 192 candidates, totalling 960 fits
[CV] END final_estimator__C=0.1, lgbm__learning_rate=0.01, lgbm__n_estimators=200, lgbm__num_leaves=40, xgb__learning_rate=0.08, xgb__max_depth=5, xgb__n_estimators=100; total time=  11.0s
[CV] END final_estimator__C=0.1, lgbm__learning_rate=0.01, lgbm__n_estimators=200, lgbm__num_leaves=40, xgb__learning_rate=

KeyboardInterrupt: 